In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 局部支持内判别方向控制：真实 MULTI V1

直接 Run all。四臂 OFF／PAYLOAD_MULTI／STATE_LOWBAND_MULTI／STATE_CONTRAST_MULTI 使用同一来源、种子、初始状态与各自新复制的原生 scheduler，各完成50步，水印作用于25–49步。原 LOWBAND16 是本次同源对照；新臂只改变 pilot 原始更新的方向。

沿用四个16×16局部支持、RM32／45状态／4年龄、alpha、eta696、cap1和原payload。在R44原active支持内，用全部公开合法候选定义 ABS 与 DIFF 联合判别空间，原始pilot先投影，再与第45窗原始更新一起执行原 shrink-only cap。空间在观测前由公开协议和密钥冻结。第45窗的实际更新会随整体截断比例变化；相同上限不代表相同实际能量。

保留100份真实写端侧车和50份状态控制方向诊断，报告实际更新量、截断及所有候选相对分数变化。方向诊断与原生终态均属写端诊断，独立于主盲接收。原生终态另保留16q／32评分及完整候选，不作为媒体检测输入。

每臂一次真实VAE解码及RGB8保存，同一份字节分别通过 DIRECT_RGB8、RAW420、MP4，再共12次VAE重编码。OLD8／LOWBAND16两个公开接收协议分别读取全部臂×通道×正确/错误密钥：48q、96评分、16704条路径成本。载荷新读24次、消息后评48条；全部接收快照先封存，再进行真值后评。重复载荷不是时间相关片段证据。

结果分支预先限定：若新臂终态仍失败，停止修补这一LOWBAND16构造并回到有终态／DIRECT正证据的OLD8；若终态形成而RAW420／MP4失败，再研究通道存留；若MP4获得可复现的正确路径优势，再推进未知相位及真实编辑。Notebook只运行当前固定实验，不自动启动后续分支。

EXECUTION_COMPLETE只表示执行完整。MSE降低、同一cap或重复载荷零误码均不表示同步成功；质量仅报告保存RGB字节的18组成对诊断。当前Python/Torch及已验证的运行方式优先，缺少实际必需组件时才修复。真实模型运行由用户执行。Published source: f98d05160d3b00b66dbd71dc088347e2c931fb7c.


In [ ]:
SOURCE_SHA = 'f98d05160d3b00b66dbd71dc088347e2c931fb7c'
FIXED = {'source_cases': 1, 'arms': 4, 'channels': 3, 'reader_families': 2, 'generated': 4, 'generation_steps': 200, 'controlled_steps': 75, 'writer_sidecars': 100, 'underlying_observations': 12, 'normalized': 12, 'keys': 2, 'keyed_projections': 48, 'path_score_records': 96, 'valid_paths': 174, 'catalog_per_record': 3915, 'path_costs': 16704, 'difference_edge_costs': 272448, 'absolute_local_state_costs': 95040, 'path_posthoc': 96, 'difference_edge_posthoc': 48, 'payload_reads': 24, 'payload_evaluations': 48, 'catalog_slots': 375840, 'structural_exclusions': 359136, 'quality_pairs': 18}
DIRECTION_FIXED = {'records': 50, 'modes_per_record': 2, 'candidates_per_mode': 174, 'directions_per_record': 3, 'candidate_effect_values': 52200}
TERMINAL_FIXED = {'inputs': 4, 'keyed_projections': 16, 'path_score_records': 32, 'path_costs': 5568, 'difference_edge_costs': 90816, 'absolute_local_state_costs': 31680, 'path_posthoc': 32, 'difference_edge_posthoc': 16, 'catalog_slots': 125280, 'structural_exclusions': 119712}
from pathlib import Path
import datetime,hashlib,json,os,subprocess,sys,traceback
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Video-Local-Fourier-RM-Contrast-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-LOCAL-FOURIER-RM-CONTRAST-'+STAMP)
PYTHON=sys.executable
ARMS=('OFF','PAYLOAD_MULTI','STATE_LOWBAND_MULTI','STATE_CONTRAST_MULTI');KEYS=('CORRECT','WRONG')
CHANNELS=('DIRECT_RGB8','RAW420','MP4');FAMILIES=('OLD8','LOWBAND16');MODES=('ABSOLUTE_CONTROL','ADJACENT_DIFFERENCE')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED,
 generation={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},writer_diagnostics={a+'/'+str(i):{'status':'NOT_RUN_SETUP'} for a in ARMS for i in range(25,50)},
 terminal_inputs={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},rasters={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},
 transport={a+'/'+c:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS},normalized={a+'/'+c:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS},
 projections={a+'/'+c+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS},
 mode_reads={a+'/'+c+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP','accepted_payload':False,'state_path_accepted':False} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS for m in MODES},
 path_posthoc={a+'/'+c+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS for m in MODES},
 edge_posthoc={a+'/'+c+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS},
 payload_reads={a+'/'+c+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS},
 payload_evaluations={a+'/'+c+'/'+k+'/'+t:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS for t in ('REGISTERED','WRONG_MESSAGE')},
 quality={c+'/'+a+'_vs_'+b:{'status':'NOT_RUN_SETUP'} for c in CHANNELS for i,a in enumerate(ARMS) for b in ARMS[:i]})

setup['direction_diagnostics']={a+'/'+str(i):{'status':'NOT_RUN_SETUP'} for a in ('STATE_LOWBAND_MULTI','STATE_CONTRAST_MULTI') for i in range(25,50)}
setup['direction_calls']={}
setup['direction_diagnostic_fixed']=DIRECTION_FIXED
setup['terminal_diagnostic_fixed']=TERMINAL_FIXED
setup['space_recipe']={'status':'NOT_RUN_SETUP','truth_inputs':False}
setup['terminal_diagnostics']=dict(
 inputs={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},
 projections={a+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for f in FAMILIES for k in KEYS},
 mode_reads={a+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP','accepted_payload':False,'state_path_accepted':False} for a in ARMS for f in FAMILIES for k in KEYS for m in MODES},
 path_posthoc={a+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP'} for a in ARMS for f in FAMILIES for k in KEYS for m in MODES},
 edge_posthoc={a+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for f in FAMILIES for k in KEYS},
 counts={},calls={},role='writer diagnostic only; never primary blind input')
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Fixed output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind published source SHA before Run all');failed('SOURCE_BINDING',error);raise error


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    import shutil
    tools={name:shutil.which(name) for name in ('ffmpeg','ffprobe')}
    tool_codes={name:logged([path,'-version'],name.upper()+'_PROBE',check=False) if path else None for name,path in tools.items()}
    if any(code!=0 for code in tool_codes.values()):
        try:
            logged(['apt-get','update'],'MEDIA_TOOL_UPDATE',check=False)
            logged(['apt-get','install','-y','ffmpeg'],'MEDIA_TOOL_REPAIR',check=False)
        except Exception as repair_error:print('Media-tool repair error retained:',repair_error)
    setup['media_tool_probe_returncodes']=tool_codes
    torch_ok=logged([PYTHON,'-c','import torch; print(torch.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:logged([PYTHON,'-m','pip','install','--upgrade','torch'],'TORCH_REPAIR',check=False)
    vae_probe='from diffusers import AutoencoderKLWan,WanPipeline; print(AutoencoderKLWan.__name__,WanPipeline.__name__)'
    vae_ok=logged([PYTHON,'-c',vae_probe],'VAE_IMPORT_PROBE',check=False)==0
    if not vae_ok:
        logged([PYTHON,'-m','pip','install','diffusers>=0.35.1,<0.40','transformers>=4.45,<5','accelerate>=0.34','safetensors>=0.4.5'],'VAE_DEPENDENCY_REPAIR',check=False)
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual VAE import/load determines availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    environment_code='import json,torch; from experiments.wan_state_clock.video_local_fourier_rm_contrast_v1_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',environment_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_separate_generation_media_workers')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_local_fourier_rm_contrast_v1_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; setup denominator retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'])
print('Calls:',result['calls'],'Workers:',result['workers'])
print('Shared before25:',result.get('before_step25_identity'))
space=result.get('space_recipe',{});receipt=space.get('receipt',{})
print('Frozen space: status=',space.get('status'),'rank=',receipt.get('rank'),'recipe=',space.get('path'),'basis=',space.get('basis_path'))
print('Direction counts/calls:',result.get('direction_counts'),result.get('direction_calls'))
for name,row in result.get('direction_diagnostics',{}).items():
    if name.endswith(('/25','/49')):
        print('Direction diagnostic:',name,'status=',row.get('status'),'source=',row.get('source'),'candidate effect values=',row.get('candidate_effect_values'))
terminal=result.get('terminal_diagnostics',{})
print('Writer terminal diagnostic counts/calls:',terminal.get('counts'),terminal.get('calls'))
for name,row in terminal.get('path_posthoc',{}).items():
    print('Writer terminal only:',name,row['status'],'rank=',row.get('true_path_rank'),'Delta=',row.get('true_path_delta'),'matches=',row.get('canonical_path_matches'))

for arm,row in result['generation'].items():print('Writer:',arm,row['status'],(row.get('receipt') or {}).get('control_norm_statistics'),(row.get('receipt') or {}).get('writer_diagnostics'))
for family in FAMILIES:
    print('Independent reader family:',family)
    for name,row in result['path_posthoc'].items():
        if '/'+family+'/' in name:print(name,row['status'],'rank=',row.get('true_path_rank'),'Delta=',row.get('true_path_delta'),'matches=',row.get('canonical_path_matches'))
for name,row in result['payload_evaluations'].items():print('Repeated payload:',name,row['status'],'bit errors=',row.get('bit_errors'),'exact=',row.get('exact_bits'))
print('Saved-RGB quality diagnostics:',result['quality'])
from IPython.display import Video,display
for arm in ARMS:
    video=result['transport'].get(arm+'/MP4',{}).get('events',{}).get('mp4',{})
    path=video.get('path')
    if video.get('status')=='SAVED' and isinstance(path,str) and Path(path).is_file():
        try:print('Saved MP4 preview:',arm);display(Video(str(path),embed=True))
        except Exception as preview_error:print('Preview unavailable; result retained:',arm,preview_error)
print('Failures:',result['failures'])
print('Evidence ceiling:',result['evidence_ceiling'])
print('Retained result:',RESULT_PATH)
